# Chapter 22: Shortest Paths

The programs from Chapter 22 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Dijkstra's Algorithm

`shortest.py`

In [1]:
"""Shortest paths in weighted graphs.

Graphs are dicts: vertex -> {neighbor: edge weight}.
"""

import heapq
import math


def dijkstra(graph, source):
    """Shortest distances from source when no weight is negative, and
    each vertex's parent on its shortest path.

    A heap hands out the closest vertex not yet finished. When a vertex
    comes off the heap its distance is final: any other route would
    pass through a vertex at least as far away, plus a weight >= 0.
    """
    dist, parent = {source: 0}, {source: None}
    heap = [(0, source)]
    done = set()
    while heap:
        d, v = heapq.heappop(heap)
        if v in done:
            continue  # an old, longer entry for v: skip it
        done.add(v)
        for w, weight in graph[v].items():
            if d + weight < dist.get(w, math.inf):
                dist[w] = d + weight
                parent[w] = v
                heapq.heappush(heap, (dist[w], w))
    return dist, parent


def path_to(parent, goal):
    path = [goal]
    while parent[path[-1]] is not None:
        path.append(parent[path[-1]])
    return path[::-1]


def bellman_ford(graph, source):
    """Shortest distances even with negative weights: relax every edge
    V - 1 times. Raises ValueError if a negative cycle can be reached,
    because then there is no shortest path."""
    dist, parent = {source: 0}, {source: None}
    for _ in range(len(graph) - 1):
        changed = False
        for v in list(dist):
            for w, weight in graph[v].items():
                if dist[v] + weight < dist.get(w, math.inf):
                    dist[w] = dist[v] + weight
                    parent[w] = v
                    changed = True
        if not changed:
            break  # nothing improved: already finished
    for v in dist:
        for w, weight in graph[v].items():
            if dist[v] + weight < dist[w]:
                raise ValueError("negative cycle")
    return dist, parent

`road_demo.py`

In [2]:
"""The shortest drive between towns (fictional, in kilometers)."""

from shortest import dijkstra, path_to

roads = [
    ("Ashford", "Brookton", 12),
    ("Ashford", "Cliffside", 35),
    ("Brookton", "Cliffside", 18),
    ("Brookton", "Dunmore", 40),
    ("Cliffside", "Dunmore", 9),
    ("Dunmore", "Eastvale", 14),
    ("Cliffside", "Eastvale", 30),
]
graph = {}
for a, b, km in roads:
    graph.setdefault(a, {})[b] = km
    graph.setdefault(b, {})[a] = km

dist, parent = dijkstra(graph, "Ashford")
for town in sorted(dist, key=dist.get):
    route = " -> ".join(path_to(parent, town))
    print(f"{town:>9} {dist[town]:>3} km: {route}")

  Ashford   0 km: Ashford
 Brookton  12 km: Ashford -> Brookton
Cliffside  30 km: Ashford -> Brookton -> Cliffside
  Dunmore  39 km: Ashford -> Brookton -> Cliffside -> Dunmore
 Eastvale  53 km: Ashford -> Brookton -> Cliffside -> Dunmore -> Eastvale


## When Weights Go Negative

`negative_demo.py`

In [3]:
"""Why Dijkstra needs weights >= 0. A negative edge (a rebate, say)
can make a route through a far vertex cheaper after Dijkstra has
already finalized a nearer one."""

from shortest import bellman_ford, dijkstra

g = {"s": {"a": 2, "b": 5}, "b": {"a": -4}, "a": {"t": 1}, "t": {}}
print("Dijkstra:    ", dijkstra(g, "s")[0])
print("Bellman-Ford:", bellman_ford(g, "s")[0])

Dijkstra:     {'s': 0, 'a': 1, 'b': 5, 't': 3}
Bellman-Ford: {'s': 0, 'a': 1, 'b': 5, 't': 2}


## Timing

`bench_shortest.py`

In [4]:
"""Dijkstra against Bellman-Ford on random graphs with n vertices and
4n edges."""

import random

import bench
from shortest import bellman_ford, dijkstra

rng = random.Random(2026)


def make(n):
    g = {v: {} for v in range(n)}
    for v in range(n):
        g[v][(v + 1) % n] = rng.randint(1, 100)  # a ring: all reachable
        for _ in range(3):
            g[v][rng.randrange(n)] = rng.randint(1, 100)
    return g, 0


sizes = [500, 1_000, 2_000, 4_000]
results = {
    "dijkstra": bench.doubling(dijkstra, make, sizes),
    "bellman_ford": bench.doubling(bellman_ford, make, sizes),
}
bench.table(results)
bench.chart(results, "ch22_shortest.png", "Shortest paths from one vertex")

         n        dijkstra    bellman_ford
       500          438 us          960 us
     1,000     910 us x2.1    2.62 ms x2.7
     2,000    2.25 ms x2.5    5.78 ms x2.2
     4,000    5.15 ms x2.3    17.7 ms x3.1
